In [3]:
import pandas as pd
import numpy as np
!pip install biopython
from Bio.SeqUtils.ProtParam import ProteinAnalysis

# Ekstrakcija značajki

Nakon pripreme podataka potrebno je sekvence peptida pretvoriti u oblik koji modeli strojnog učenja mogu koristiti. Budući da modeli ne mogu izravno raditi sa slovima aminokiselina, iz svake sekvence izračunavaju se numeričke vrijednosti koje opisuju njezina svojstva.

Cilj ovog koraka je dobiti skup značajki koji će što bolje opisivati pojedini peptid i omogućiti modelu da nauči razlikovati antimikrobne peptide od onih koji nemaju antimikrobno djelovanje.


In [6]:
dataset = pd.read_csv("amp_dataset.csv")

print(dataset.shape)

dataset.head()

(3344, 2)


,SEQUENCE,label
0,CFQWQRNARKVR,1
1,GYFYHTEPTSHHRFWDFFISMFPYA,0
2,WLNALLHHGLNCAKGVL,1
3,RLWRIVVIRVAR,1
4,ILGTILGLLKSL,1


## Pregled podataka

U prethodnom dijelu projekta pripremljen je konačni skup podataka koji sadrži sekvence peptida i pripadajuće oznake klasa.

Prije izračuna značajki potrebno je učitati podatke i provjeriti njihov oblik. Na taj način možemo biti sigurni da su sekvence ispravno spremljene i spremne za daljnju obradu.


In [7]:
def calculate_net_charge(sequence):

    positive = sequence.count("K") + sequence.count("R")

    negative = sequence.count("D") + sequence.count("E")

    return positive - negative

## Izračun značajki

Za svaku sekvencu računaju se različita svojstva koja mogu biti korisna za klasifikaciju.

Prvo se računaju osnovne značajke poput duljine sekvence, neto naboja, hidrofobnosti, molekularne mase, aromatičnosti i izoelektrične točke. Ove vrijednosti opisuju kemijska i fizikalna svojstva peptida.

Osim toga računa se i aminokiselinski sastav (AAC), odnosno udio svake aminokiseline unutar sekvence. Time model dobiva dodatne informacije o strukturi peptida koje mogu pomoći u razlikovanju pojedinih klasa.


In [8]:
def extract_features(sequence):

    protein = ProteinAnalysis(sequence)

    features = {}

    features["length"] = len(sequence)

    features["net_charge"] = calculate_net_charge(
        sequence
    )

    features["hydrophobicity"] = (
        protein.gravy()
    )

    features["molecular_weight"] = (
        protein.molecular_weight()
    )

    features["aromaticity"] = (
        protein.aromaticity()
    )

    features["isoelectric_point"] = (
        protein.isoelectric_point()
    )

    return features

In [9]:
features_df = dataset["SEQUENCE"].apply(
    extract_features
)

features_df = pd.DataFrame(
    features_df.tolist()
)

features_df.head()

,length,net_charge,hydrophobicity,molecular_weight,aromaticity,isoelectric_point
0,12,4,-1.458333,1591.8412,0.166667,11.712568
1,25,-1,-0.412000,3184.4944,0.360000,6.260497
2,17,1,0.605882,1859.2016,0.058824,8.239656
3,12,4,0.691667,1536.9117,0.083333,11.999968
4,12,1,1.816667,1240.5746,0.000000,8.750052


In [10]:
features_df.shape

(3344, 6)

In [11]:
AA = list(
    "ACDEFGHIKLMNPQRSTVWY"
)

def calculate_aac(sequence):

    protein = ProteinAnalysis(
        sequence
    )

    return protein.get_amino_acids_percent()

In [13]:
AA = list("ACDEFGHIKLMNPQRSTVWY")

def calculate_aac(sequence):

    protein = ProteinAnalysis(sequence)

    aa_percent = protein.amino_acids_percent

    return aa_percent

In [14]:
aac_df = dataset["SEQUENCE"].apply(
    calculate_aac
)

aac_df = pd.DataFrame(
    aac_df.tolist()
)

aac_df.head()

,A,C,D,E,F,G,H,I,K,L,M,N,P,Q,R,S,T,V,W,Y
0,8.333333,8.333333,0.0,0.0,8.333333,0.000000,0.000000,0.000000,8.333333,0.000000,0.0,8.333333,0.0,16.666667,25.000000,0.000000,0.000000,8.333333,8.333333,0.0
1,4.000000,0.000000,4.0,4.0,20.000000,4.000000,12.000000,4.000000,0.000000,0.000000,4.0,0.000000,8.0,0.000000,4.000000,8.000000,8.000000,0.000000,4.000000,12.0
2,11.764706,5.882353,0.0,0.0,0.000000,11.764706,11.764706,0.000000,5.882353,29.411765,0.0,11.764706,0.0,0.000000,0.000000,0.000000,0.000000,5.882353,5.882353,0.0
3,8.333333,0.000000,0.0,0.0,0.000000,0.000000,0.000000,16.666667,0.000000,8.333333,0.0,0.000000,0.0,0.000000,33.333333,0.000000,0.000000,25.000000,8.333333,0.0
4,0.000000,0.000000,0.0,0.0,0.000000,16.666667,0.000000,16.666667,8.333333,41.666667,0.0,0.000000,0.0,0.000000,0.000000,8.333333,8.333333,0.000000,0.000000,0.0


In [15]:
aac_df.shape

(3344, 20)

In [16]:
X = pd.concat(
    [
        features_df,
        aac_df
    ],
    axis=1
)

X.head()

,length,net_charge,hydrophobicity,molecular_weight,aromaticity,isoelectric_point,A,C,D,E,...,M,N,P,Q,R,S,T,V,W,Y
0,12,4,-1.458333,1591.8412,0.166667,11.712568,8.333333,8.333333,0.0,0.0,...,0.0,8.333333,0.0,16.666667,25.000000,0.000000,0.000000,8.333333,8.333333,0.0
1,25,-1,-0.412000,3184.4944,0.360000,6.260497,4.000000,0.000000,4.0,4.0,...,4.0,0.000000,8.0,0.000000,4.000000,8.000000,8.000000,0.000000,4.000000,12.0
2,17,1,0.605882,1859.2016,0.058824,8.239656,11.764706,5.882353,0.0,0.0,...,0.0,11.764706,0.0,0.000000,0.000000,0.000000,0.000000,5.882353,5.882353,0.0
3,12,4,0.691667,1536.9117,0.083333,11.999968,8.333333,0.000000,0.0,0.0,...,0.0,0.000000,0.0,0.000000,33.333333,0.000000,0.000000,25.000000,8.333333,0.0
4,12,1,1.816667,1240.5746,0.000000,8.750052,0.000000,0.000000,0.0,0.0,...,0.0,0.000000,0.0,0.000000,0.000000,8.333333,8.333333,0.000000,0.000000,0.0


In [17]:
X["label"] = dataset["label"]

In [18]:
X.shape

(3344, 27)

In [19]:
X.isnull().sum().sum()

np.int64(0)

In [20]:
X.to_csv(
    "feature_matrix.csv",
    index=False
)

print("Feature matrix spremljen.")

Feature matrix spremljen.


In [21]:
print(X.shape)

X.head()

(3344, 27)


,length,net_charge,hydrophobicity,molecular_weight,aromaticity,isoelectric_point,A,C,D,E,...,N,P,Q,R,S,T,V,W,Y,label
0,12,4,-1.458333,1591.8412,0.166667,11.712568,8.333333,8.333333,0.0,0.0,...,8.333333,0.0,16.666667,25.000000,0.000000,0.000000,8.333333,8.333333,0.0,1
1,25,-1,-0.412000,3184.4944,0.360000,6.260497,4.000000,0.000000,4.0,4.0,...,0.000000,8.0,0.000000,4.000000,8.000000,8.000000,0.000000,4.000000,12.0,0
2,17,1,0.605882,1859.2016,0.058824,8.239656,11.764706,5.882353,0.0,0.0,...,11.764706,0.0,0.000000,0.000000,0.000000,0.000000,5.882353,5.882353,0.0,1
3,12,4,0.691667,1536.9117,0.083333,11.999968,8.333333,0.000000,0.0,0.0,...,0.000000,0.0,0.000000,33.333333,0.000000,0.000000,25.000000,8.333333,0.0,1
4,12,1,1.816667,1240.5746,0.000000,8.750052,0.000000,0.000000,0.0,0.0,...,0.000000,0.0,0.000000,0.000000,8.333333,8.333333,0.000000,0.000000,0.0,1


## Zaključak

Nakon izračuna svih značajki dobiven je novi skup podataka koji više ne sadrži samo sekvence, nego numerički opis svakog peptida.

Takav oblik podataka prikladan je za modele strojnog učenja te će se koristiti u sljedećem koraku projekta za treniranje i usporedbu različitih modela klasifikacije.
